## Installing required libraries

In [10]:
# !pip install -q "numpy<2.0" "transformers>=4.40.0" chromadb llama-index llama-index-embeddings-huggingface pypdf

In [11]:
# !pip uninstall -y chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp protobuf
# !pip install chromadb==0.5.5 \
#     opentelemetry-api==1.27.0 \
#     opentelemetry-sdk==1.27.0 \
#     opentelemetry-exporter-otlp==1.27.0 \
#     protobuf==4.25.3 -q

In [12]:
# !pip install -U chromadb==0.5.23 llama-index-vector-stores-chroma==0.5.5 -q

## Chunking and Storing

In [13]:
#initiating a client and collection 
import chromadb

db         = chromadb.PersistentClient(path='./chroma_db')
collection = db.get_or_create_collection('whitepapers')

In [14]:
#connect it with llamaindex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context      = StorageContext.from_defaults(vector_store=vector_store)

Now llamaindex writes into chromadb directly 

In [15]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset = Path('/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs')

loader = PDFReader()

documents = []

for pdf in dataset.glob('*.pdf'):
    doc = loader.load_data(file=pdf)
    documents.extend(doc)

print(documents[0].text[:500])

Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



In [16]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex(
    documents,
    storage_context=storage_context
)

2026-05-28 18:19:07,061 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-28 18:19:07,078 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-05-28 18:19:07,138 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-28 18:19:07,155 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-28 18:19:07,157 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-05-28 18:19:07,216 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-28 18:19:08,113 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-28 18:19:08,189 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-28 18:19:08,253 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-28 18:19:08,315 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-28 18:19:08,377 - INFO - HTTP Request: HEAD 

In [17]:
print(collection.count()) #number of nodes/chunks stored in the db

260


In [19]:
for doc in documents:
    print(doc.metadata)
    break

{'page_label': '1', 'file_name': '22365_19_Agents_v8.pdf'}


In [24]:
collection.get(include=["documents"],where={'page_label':'23'}) #this is simply us querying with the db

{'ids': ['db210844-aa79-4235-97d0-13091f8245e6',
  '9b6d6228-74ce-45e2-9296-79f011d369fa',
  '0709e6f1-e587-4ec8-abda-6ee217c89ba2',
  'c83ef859-2a9d-49b2-ab44-6d9c48858d3a'],
 'embeddings': None,
 'documents': ["Agents\n23\nFebruary 2025\n \nFigure 9. Sequence diagram showing the lifecycle of a Function Call\nThe result of the example in Figure 9 is that the model is leveraged to “fill in the blanks” with \nthe parameters required for the Client side UI to make the call to the Google Places API. The \nClient side UI manages the actual API call using the parameters provided by the model in the \nreturned Function. This is just one use case for Function Calling, but there are many other \nscenarios to consider like:\n• You want a language model to suggest a function that you can use in your code, but you \ndon't want to include credentials in your code. Because function calling doesn't run the \nfunction, you don't need to include credentials in your code with the function information."